# GRACE H1 — thử preparation trên WSL `/home`

Notebook này chạy phần CPU/RAM của ReVeal trong thư mục Linux local `/home`, với cấu hình hiện tại `graph_workers=3`. Không chạy Kaggle, GPU, inference hoặc gọi LLM.

Mở notebook từ bản project trong `/home/yungn/GRACE-H1-SecurityProjection-reveal-home-test/`. Cell đầu nhập CPU inputs và trained heads, dùng Joern 4.0.623 đã có trong WSL, và nhập checkpoint tùy chọn. Nếu muốn dùng checkpoint Colab mới nhất, tải `grace-h1-cpu-checkpoint.zip` từ Drive và đặt vào thư mục `bundles/` trước khi chạy.

Kết quả chuẩn bị và checkpoint local được lưu trong `bundles/`. Theo dõi RAM bằng `free -h` trong WSL; mỗi worker có thể chạy một JVM Joern với heap tối đa 2 GB.

In [1]:
from pathlib import Path, PurePosixPath
import sys, zipfile, shutil, json, os, subprocess
ROOT=Path('/home/yungn/GRACE-H1-SecurityProjection-reveal-home-test')
BUNDLES=ROOT/'bundles'; BUNDLES.mkdir(parents=True,exist_ok=True)
CPU_INPUTS=BUNDLES/'grace-h1-cpu-inputs.zip'
JOERN_HOME=Path('/home/yungn/tools/joern/y/joern-cli')
HEADS=BUNDLES/'grace-h1-trained-heads.zip'
CHECKPOINT=BUNDLES/'grace-h1-cpu-checkpoint.zip'
for p in (CPU_INPUTS,HEADS):
    if not p.is_file(): raise FileNotFoundError(f'Missing bundle: {p}')
if not (ROOT/'grace_h1').is_dir():
    staging=ROOT/'_cpu_input_bootstrap'; staging.mkdir(exist_ok=True)
    with zipfile.ZipFile(CPU_INPUTS) as z:
        names=z.namelist()
        for name in names:
            rel=PurePosixPath(name)
            if rel.is_absolute() or '..' in rel.parts or ':' in name or chr(92) in name: raise ValueError(f'Unsafe bundle path: {name}')
        z.extractall(staging)
    sys.path.insert(0,str(staging))
    from grace_h1.portable import import_bundle as _import_cpu_inputs
    print(_import_cpu_inputs(ROOT,CPU_INPUTS,'cpu_inputs'))
    shutil.rmtree(staging); sys.path.remove(str(staging))
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from grace_h1.portable import import_bundle, create_bundle
from grace_h1.io import read_json
if not (JOERN_HOME/'joern-parse').is_file() or not (JOERN_HOME/'joern-export').is_file(): raise FileNotFoundError(f'Joern installation missing: {JOERN_HOME}')
os.environ['JOERN_HOME']=str(JOERN_HOME)
print(import_bundle(ROOT,HEADS,'trained_heads'))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'cpu_checkpoint'))
config=read_json(ROOT/'configs'/'experiment.json')
assert config['datasets']==['reveal'] and config['pipeline']['graph_workers']==3, config
print('ROOT:',ROOT,'| graph_workers:',config['pipeline']['graph_workers'])
print(subprocess.run(['free','-h'],capture_output=True,text=True).stdout)

{'kind': 'trained_heads', 'copied': 0, 'reused_identical': 11, 'archive_sha256': '5800350161584e3b1fb3737669538fb650dad0d05d8a975623cf96e43aa30162'}
ROOT: /home/yungn/GRACE-H1-SecurityProjection-reveal-home-test | graph_workers: 3
               total        used        free      shared  buff/cache   available
Mem:            13Gi        10Gi       123Mi        16Mi       3.0Gi       2.9Gi
Swap:          4.0Gi       2.6Gi       1.4Gi



In [2]:
from grace_h1.offload import require_trained_heads
from grace_h1.experiment import prepare_all
selection=require_trained_heads(ROOT,config)
def save_checkpoint():
    info=create_bundle(ROOT,CHECKPOINT,'cpu_checkpoint')
    print('Local checkpoint:',info['path'],'bytes:',info['bytes'],'sha256:',info['sha256'],flush=True)
try:
    runs=prepare_all(ROOT,config,checkpoint_callback=save_checkpoint)
finally:
    save_checkpoint()
from grace_h1.offload import require_prepared_runs
print('Prepared runs:',require_prepared_runs(ROOT,config))
import pandas as pd
timings=[]
for row in runs:
    manifest=read_json(ROOT/row['path']/'run_manifest.json')
    timings.append({'dataset':row['dataset'],'split':row['split'],'arm':row['arm'],'graph_workers':manifest['config']['graph_workers'],'graph_reused':manifest['graph_reused'],'graph_built':manifest['graph_built'],'prepare_seconds':round(manifest['prepare_seconds'],1)})
display(pd.DataFrame(timings))
result=create_bundle(ROOT,BUNDLES/'grace-h1-cpu-prepared-wsl.zip','cpu_prepared')
print('Prepared bundle:',result)

reveal/test common candidates 250/2274 (0s)
reveal/test common candidates 500/2274 (0s)
reveal/test common candidates 750/2274 (0s)
reveal/test common candidates 1000/2274 (0s)
reveal/test common candidates 1250/2274 (0s)
reveal/test common candidates 1500/2274 (0s)
reveal/test common candidates 1750/2274 (0s)
reveal/test common candidates 2000/2274 (0s)
reveal/test common candidates 2250/2274 (0s)
reveal/baseline_whitening: 9819 unique graph features required
graphs 500/9819
Local checkpoint: /home/yungn/GRACE-H1-SecurityProjection-reveal-home-test/bundles/grace-h1-cpu-checkpoint.zip bytes: 25159234 sha256: d8272c27ab80400bd070f788e154443667d56df637c0189b20911fc18d0b7d6f
graphs 1000/9819
Local checkpoint: /home/yungn/GRACE-H1-SecurityProjection-reveal-home-test/bundles/grace-h1-cpu-checkpoint.zip bytes: 26190024 sha256: 05cb5459c2d58e0cf73b1cbb31b5d019a7687a5454b4cae1f82b859feb686997
graphs 1500/9819
Local checkpoint: /home/yungn/GRACE-H1-SecurityProjection-reveal-home-test/bundles/gr

,dataset,split,arm,graph_workers,graph_reused,graph_built,prepare_seconds
0,reveal,test,baseline_whitening,3,1151,8668,25765.1
1,reveal,test,h1_supcon_hard,3,8356,1224,3615.9


Prepared bundle: {'path': '/home/yungn/GRACE-H1-SecurityProjection-reveal-home-test/bundles/grace-h1-cpu-prepared-wsl.zip', 'kind': 'cpu_prepared', 'files': 9910, 'bytes': 127043247, 'sha256': '3cdc70a77d755b3cc12912d63b93d012db7faeab297fbe5cd0343b23c307d6c4', 'job': {'data_manifest_sha256': 'a3f86cbf6b3ac9aff681ac890711e7600e7546d52dc7fc55a73632bab902b0c4', 'experiment': {'bootstrap': 1000, 'datasets': ['reveal'], 'inference_workers': 4, 'llm_arms': ['baseline_whitening', 'h1_supcon_hard'], 'objectives': ['ce', 'supcon_hard'], 'pipeline': {'graph_workers': 3, 'joern_home': None, 'lexical_weight': 0.7, 'semantic_top_k': 5, 'whitening_dimension': 256}, 'splits': ['test'], 'training': {'batch_size': 256, 'class_weighted_ce': True, 'device': 'auto', 'dimension': 128, 'epochs': 30, 'hard_negative_weight': 0.1, 'learning_rate': 0.001, 'margin': 0.2, 'patience': 6, 'seed': 42, 'supcon_weight': 0.1, 'temperature': 0.1, 'weight_decay': 0.0001}}, 'source': {'__init__.py': '16c7a88f18db7d7c2cf76